# Lecture 6 - pandas II

**Lecture Topic**: The course project, then more pandas: summary statistics, group-by, cleaning, merging, and some of the ways pandas can go silently wrong


## Announcements

- Today we'll start with the course project. Please start forming groups!
- ClassBuzz poll for today: `D3JJ`

## The course project

The project is where you'll put all of this into practice. In groups of 3-4, you'll take a data science problem from a loose idea to a working, tested and documented piece of software, built with whatever AI tools you have access to (or none).

It's worth 35% of your grade, and runs alongside the lectures for the rest of the quarter.

It's up to you what you work on. It could be:
 - **research based**: something you need for your own research, for example a pipeline to clean and combine data from your lab's instruments
 - **class based**: an analysis from another course, taken further and built properly
 - **a hobby project**: for example, analysing your own running or cycling data, or a tool that tracks surf conditions from NOAA buoy data

It should be meaty enough to be useful: real data (ideally more than fits comfortably in a spreadsheet), more than one processing step, and a clear answer to "who would use this, and for what?"

### What's involved

Each milestone uses what we'll have covered in the lectures by then:

| Milestone | Due | What |
|---|---|---|
| M1 - Proposal | Fri Oct 16 | Your group, your idea and your data (next slide) |
| M2 - Design | Mon Nov 16 | A spec, the architecture, how each step scales (measured, not guessed), a test plan and a working skeleton |
| M3 - Final | Sun Nov 29 | The repository (code, tests and CI), a log of how you used AI tools, and a short report |
| Presentation and interview | Nov 30 - Dec 4 | A short presentation in class, and a 10 minute interview with your group where everyone answers questions |

You can use any coding assistant or agent you like, or none. What we grade is your judgement: whether you can show the result is correct, explain why it's built the way it is, and say when it's done. "The agent wrote it" won't get you very far in the interview!

### Milestone 1: the proposal

Form a group of 3-4 and submit a short proposal (about a page) on Gradescope, one per group, by **Friday Oct 16 at 11:59pm**. It should cover:

 - who is in the group
 - the problem: what you want to build, and who it's for (that can be you!)
 - the data: where it comes from, roughly how big it is, and whether you already have access to it
 - what "done" looks like: 3-5 things the finished project should be able to do
 - anything we should know about, for example if the data is sensitive or the project overlaps with your research

If you're having trouble finding a group, let me know by Wednesday Oct 14 and I'll help match people up.

I'll give feedback on every proposal, and it's fine to adjust the idea after that.

## Recap

In the last lecture, we covered the following topics:
 - Broadcasting
 - Views vs copies
 - A first look at cost: Python loops vs NumPy
 - pandas Series, DataFrames and the Index
 - Inspecting a dataset
 - Selecting data with `[]`, `loc`, `iloc`, Boolean masks and `query()`
 - Index alignment

Today, we will cover the following topics:
 - The course project
 - Index alignment (one more example)
 - Summary statistics and group-by
 - Cleaning data: renaming, dates, new columns and sorting
 - Merging
 - Some silent failure modes: types and chained assignment

## Index alignment, continued

Let's pick up where we left off on Monday. First, let's load the taxi data again:

In [ ]:
import pandas as pd

taxis = pd.read_csv('../data/2019_Yellow_Taxi_Trip_Data.csv')

### Quick quiz: spot the bug

Tips are only recorded for card payments, so I asked an agent to add a column with the tip percentage for card trips. It wrote:

In [ ]:
card = taxis[taxis.payment_type == 1]
taxis['tip_pct'] = card.tip_amount / card.fare_amount * 100

What does `tip_pct` contain for the trips paid in cash?

 - A. `0`
 - B. `NaN`
 - C. The card values, shifted up to fill the gaps
 - D. It raises an error

In [ ]:
cash = taxis.payment_type == 2
taxis.loc[cash, ['payment_type', 'tip_amount', 'tip_pct']].head()

In [ ]:
print(len(taxis), taxis.tip_pct.count())
print(taxis.tip_pct.mean())

The column was lined up on the index, so every row that wasn't in `card` got `NaN`. `mean()` then quietly skips the `NaN`s, so the mean is just the card trips.

Is that a bug? It depends on what you wanted: `NaN` (unknown), `0`, or leaving cash trips out altogether are all reasonable choices, and they give different answers. The agent made that choice for you, and didn't mention it.

Note, if you convert to NumPy with `.to_numpy()` (or `.values`), the index is thrown away and alignment no longer happens. We'll see some more of these silent failure modes later today.

## Calculating summary statistics

We can already extract some interesting insights from the taxi data by calculating summary statistics.

#### How were the trips paid for?

In [ ]:
taxis.payment_type.value_counts()

*Tip: Pass in `normalize=True` to see this result as proportions. Check the [documentation](https://pandas.pydata.org/docs/reference/api/pandas.Series.value_counts.html) for additional functionality.*

(Payment type 1 is a card, 2 is cash, 3 is no charge and 4 is a dispute.)

### Quick quiz

Which is larger for this data: the mean fare or the median fare?

 - A. The mean
 - B. The median
 - C. They're about the same

In [ ]:
print(taxis.fare_amount.mean())
print(taxis.fare_amount.median())

**Important**: The mean isn't always the best measure of central tendency. If there are outliers in the distribution, the mean will be skewed. Here, a few long (airport) trips pull the mean up, and the median is a better measure of a typical fare.

## Group-by operations

Rather than perform aggregations, like `mean()` or `describe()`, on the full dataset at once, we can perform these calculations per group by first calling `groupby()`:

In [ ]:
taxis.groupby('payment_type').tip_amount.mean()

Notice the tip is zero for everything except card payments. Cash tips aren't recorded at all, which is why the agent filtered on card payments earlier.

We can calculate several things at once with `agg()`, and give the results sensible names:

In [ ]:
taxis.groupby('payment_type').agg(
    trips=('fare_amount', 'size'),
    mean_fare=('fare_amount', 'mean'),
    mean_tip=('tip_amount', 'mean'),
)

We are only scratching the surface; some additional functionalities to be aware of include the following:
- We can group by multiple columns, which creates a hierarchical index.
- `pivot_table()` does a group-by over two columns at once and lays the result out as a table.
- We can group by date ranges with `resample()` (or a `pd.Grouper()` object).

## Data cleaning

In this section, we will take a look at renaming columns, type conversion, creating new columns and sorting, all of which make our analysis easier.

### Renaming columns

Let's start by renaming the datetime columns:

In [ ]:
taxis = taxis.rename(
    columns={
        'tpep_pickup_datetime': 'pickup',
        'tpep_dropoff_datetime': 'dropoff'
    }
)
taxis.columns

### Type conversion

On Monday we saw that `pickup` and `dropoff` are stored as strings:

In [ ]:
taxis[['pickup', 'dropoff']].dtypes

They should be stored as datetimes. Let's fix this:

In [ ]:
taxis[['pickup', 'dropoff']] = taxis[['pickup', 'dropoff']].apply(pd.to_datetime)
taxis[['pickup', 'dropoff']].dtypes

### Creating new columns

Let's calculate the elapsed time of each trip, and its average speed:

In [ ]:
taxis = taxis.assign(
    elapsed_time=lambda x: x.dropoff - x.pickup,
    avg_speed=lambda x: x.trip_distance / (x.elapsed_time.dt.total_seconds() / 60 / 60),
)
taxis[['pickup', 'dropoff', 'elapsed_time', 'trip_distance', 'avg_speed']].head(3)

Some things to note:
- We used `lambda` functions so that we can use the `elapsed_time` column in the same call that creates it.
- To create a single new column, we can also use `df['new_col'] = <values>`.

### Sorting by values

We can use the `sort_values()` method to sort based on any number of columns, and `nlargest()` / `nsmallest()` to pick out the largest/smallest rows. Let's look at the longest trips:

In [ ]:
taxis.nlargest(3, 'elapsed_time')[['pickup', 'dropoff', 'elapsed_time', 'trip_distance', 'fare_amount']]

Question: what do you think happened here?

A trip of less than a mile that takes 23 hours and 59 minutes is almost certainly a data problem rather than a very slow taxi. These are the sorts of things that are worth looking for before you trust an average (or an agent's summary of one).

## Merging

Payment types are stored as numbers, which isn't very readable. Let's add a description from a small lookup table:

In [ ]:
payment_types = pd.DataFrame({
    'payment_type': [1, 2, 3, 4],
    'description': ['Card', 'Cash', 'No charge', 'Dispute'],
})

merged = taxis.merge(payment_types, on='payment_type', how='left')
merged[['payment_type', 'description', 'fare_amount']].head()

*Tip: There are many parameters for this method, so be sure to check out the [documentation](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.merge.html). To append rows, take a look at the `pd.concat()` function.*

### Quick quiz: spot the bug

Someone has added a second description for card payments to the lookup table:

In [ ]:
payment_types = pd.DataFrame({
    'payment_type': [1, 1, 2, 3, 4],
    'description': ['Card', 'Credit card', 'Cash', 'No charge', 'Dispute'],
})

merged = taxis.merge(payment_types, on='payment_type', how='left')

How many rows does `merged` have?

 - A. 10,000
 - B. 5
 - C. 17,103
 - D. An error

In [ ]:
len(merged)

Every card trip matched *both* rows, so it now appears twice. Any total you compute from `merged` is wrong, and there's no error or warning.

When you merge, check the number of rows before and after. Or better, ask pandas to check for you:

In [ ]:
taxis.merge(payment_types, on='payment_type', how='left', validate='many_to_one')

## Silent failure modes: types

Here are some daily ridership numbers, read from a CSV file:

In [ ]:
import io

csv = io.StringIO('''day,riders
Mon,1200
Tue,950
Wed,"1,310"
Thu,1100
''')
riders = pd.read_csv(csv)
riders

### Quick quiz

What does `riders.riders.sum()` return?

 - A. `4560`
 - B. `'12009501,3101100'`
 - C. An error
 - D. `NaN`

In [ ]:
riders.riders.sum()

One value had a comma in it, so pandas read the whole column as strings, and "adding" strings sticks them together:

In [ ]:
riders.dtypes

A common "fix" (and one an agent will often reach for) is to force the column to be numeric:

In [ ]:
pd.to_numeric(riders.riders, errors='coerce').sum()

That runs, but `errors='coerce'` quietly turned Wednesday into `NaN`, and `sum()` skipped it. A better fix is to tell pandas how the numbers are written:

In [ ]:
csv.seek(0)
pd.read_csv(csv, thousands=',').riders.sum()

## Silent failure modes: chained assignment

Suppose we decide the negative fares should be set to zero, and write:

In [ ]:
taxis[taxis.fare_amount < 0]['fare_amount'] = 0

### Quick quiz

How many negative fares are left?

 - A. 0
 - B. 28
 - C. An error

In [ ]:
(taxis.fare_amount < 0).sum()

`taxis[taxis.fare_amount < 0]` makes a new DataFrame, and we set the values in that, not in `taxis`. In pandas 3 this never works, and you get a warning (in older versions it sometimes worked, which was worse!). So do read your warnings.

The fix is to do the selection and the assignment in one step with `loc`:

In [ ]:
taxis.loc[taxis.fare_amount < 0, 'fare_amount'] = 0
(taxis.fare_amount < 0).sum()

Whether setting refunds to zero is actually the right thing to do is a different question!

## Summary

Today, we covered the following topics:
 - The course project, and Milestone 1 (proposal due Friday Oct 16)
 - Index alignment, and how it can quietly fill a column with `NaN`
 - Summary statistics and group-by
 - Cleaning data: renaming, dates, new columns and sorting
 - Merging (and checking the number of rows afterwards)
 - Silent failure modes: types and chained assignment

On Friday, xarray and matplotlib. And please start forming your groups!